# Sequence-to-Sequence (Seq2Seq) with Attention Mechanisms

This notebook covers:
1. **Encoder-Decoder RNNs with Bahdanau (Additive) Attention**
2. **Encoder-Decoder RNNs with Luong (Dot-Product) Attention**

Seq2Seq models are powerful for machine translation, summarization, and question-answering tasks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =====================================================================
# BAHDANAU (ADDITIVE) ATTENTION MECHANISM
# =====================================================================

class BahdanauAttention(nn.Module):
    """
    Additive Attention (Bahdanau et al., 2014).
    score(s_t, h_i) = v^T * tanh(W_q * s_t + W_k * h_i)
    
    Args:
        hidden_dim: Dimension of the hidden state
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.query_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, query, keys, mask=None):
        """
        Args:
            query: Decoder hidden state [batch_size, hidden_dim]
            keys: Encoder outputs [batch_size, seq_len, hidden_dim]
            mask: Attention mask [batch_size, seq_len]
        
        Returns:
            context: Context vector [batch_size, hidden_dim]
            attention_weights: Attention weights [batch_size, seq_len]
        """
        # Project query and keys
        query_proj = self.query_proj(query).unsqueeze(1)  # [batch, 1, hidden]
        keys_proj = self.key_proj(keys)  # [batch, seq_len, hidden]
        
        # Compute attention scores
        scores = self.v(torch.tanh(query_proj + keys_proj))  # [batch, seq_len, 1]
        scores = scores.squeeze(-1)  # [batch, seq_len]
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Compute attention weights
        attention_weights = F.softmax(scores, dim=1)  # [batch, seq_len]
        
        # Compute context vector
        context = torch.bmm(attention_weights.unsqueeze(1), keys).squeeze(1)
        
        return context, attention_weights


# =====================================================================
# LUONG (DOT-PRODUCT) ATTENTION MECHANISM
# =====================================================================

class LuongAttention(nn.Module):
    """
    Multiplicative/Dot-Product Attention (Luong et al., 2015).
    score(s_t, h_i) = s_t^T * W * h_i
    
    Variants:
    - 'dot': No learnable weight matrix
    - 'general': Learnable weight matrix W
    - 'concat': Concatenation-based (similar to Bahdanau)
    """
    
    def __init__(self, hidden_dim, method='general'):
        super().__init__()
        self.method = method
        self.hidden_dim = hidden_dim
        
        if method == 'general':
            self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif method == 'concat':
            self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, query, keys, mask=None):
        """
        Args:
            query: Decoder hidden state [batch_size, hidden_dim]
            keys: Encoder outputs [batch_size, seq_len, hidden_dim]
            mask: Attention mask [batch_size, seq_len]
        
        Returns:
            context: Context vector [batch_size, hidden_dim]
            attention_weights: Attention weights [batch_size, seq_len]
        """
        if self.method == 'dot':
            scores = torch.bmm(keys, query.unsqueeze(-1)).squeeze(-1)  # [batch, seq_len]
        
        elif self.method == 'general':
            query_proj = self.W(query)  # [batch, hidden]
            scores = torch.bmm(keys, query_proj.unsqueeze(-1)).squeeze(-1)  # [batch, seq_len]
        
        elif self.method == 'concat':
            query_proj = self.W_q(query).unsqueeze(1)  # [batch, 1, hidden]
            keys_proj = self.W_k(keys)  # [batch, seq_len, hidden]
            scores = self.v(torch.tanh(query_proj + keys_proj)).squeeze(-1)  # [batch, seq_len]
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Compute attention weights
        attention_weights = F.softmax(scores, dim=1)  # [batch, seq_len]
        
        # Compute context vector
        context = torch.bmm(attention_weights.unsqueeze(1), keys).squeeze(1)
        
        return context, attention_weights


In [ ]:
# =====================================================================
# ENCODER-DECODER WITH ATTENTION
# =====================================================================

class Encoder(nn.Module):
    """LSTM-based Encoder"""
    
    def __init__(self, input_dim, hidden_dim, n_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, 
                           batch_first=True, dropout=dropout, bidirectional=True)
        self.fc_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_c = nn.Linear(hidden_dim * 2, hidden_dim)
    
    def forward(self, x):
        """
        Args:
            x: Input tensor [batch_size, seq_len, input_dim]
        
        Returns:
            encoder_outputs: All hidden states [batch_size, seq_len, hidden_dim*2]
            h_n, c_n: Final hidden and cell states
        """
        encoder_outputs, (h_n, c_n) = self.lstm(x)
        
        # h_n shape: [n_layers*2, batch, hidden_dim] -> combine bidirectional
        h_n = h_n.view(self.n_layers, 2, x.size(0), self.hidden_dim)
        h_n = h_n[-1]  # Last layer: [2, batch, hidden]
        c_n = c_n.view(self.n_layers, 2, x.size(0), self.hidden_dim)[-1]
        
        h_n = torch.cat([h_n[0], h_n[1]], dim=1)  # [batch, hidden*2]
        c_n = torch.cat([c_n[0], c_n[1]], dim=1)
        
        h_n = torch.tanh(self.fc_h(h_n))  # [batch, hidden]
        c_n = torch.tanh(self.fc_c(c_n))
        
        return encoder_outputs, h_n, c_n


class AttnDecoder(nn.Module):
    """LSTM-based Decoder with Attention"""
    
    def __init__(self, output_dim, hidden_dim, attention_type='bahdanau'):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        self.lstm = nn.LSTMCell(hidden_dim, hidden_dim)
        
        # Initialize attention
        if attention_type == 'bahdanau':
            self.attention = BahdanauAttention(hidden_dim)
        else:
            self.attention = LuongAttention(hidden_dim, method='general')
        
        self.fc_out = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, decoder_input, h, c, encoder_outputs):
        """
        Args:
            decoder_input: [batch_size, hidden_dim]
            h, c: Hidden and cell states [batch_size, hidden_dim]
            encoder_outputs: [batch_size, src_len, hidden_dim*2]
        
        Returns:
            output: Prediction logits [batch_size, output_dim]
            h, c: Updated states
            attention_weights: Attention distribution
        """
        h, c = self.lstm(decoder_input, (h, c))
        
        context, attention_weights = self.attention(h, encoder_outputs)
        
        combined = torch.cat([h, context], dim=1)
        output = self.fc_out(self.dropout(combined))
        
        return output, h, c, attention_weights


In [ ]:
# =====================================================================
# COMPLETE SEQ2SEQ MODEL WITH ATTENTION
# =====================================================================

class Seq2Seq(nn.Module):
    """Complete Seq2Seq model with encoder-decoder architecture and attention"""
    
    def __init__(self, input_dim, output_dim, hidden_dim, 
                 attention_type='bahdanau', device='cpu'):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim)
        self.decoder = AttnDecoder(output_dim, hidden_dim, attention_type)
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        Args:
            src: Source sequence [batch_size, src_len, input_dim]
            trg: Target sequence [batch_size, trg_len, input_dim]
            teacher_forcing_ratio: Probability of using ground truth in decoder
        
        Returns:
            predictions: [batch_size, trg_len, output_dim]
            attention_weights: Attention visualizations
        """
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        
        # Encode
        encoder_outputs, h, c = self.encoder(src)
        
        # Initialize decoder
        decoder_input = trg[:, 0, :]
        predictions = []
        all_attention = []
        
        # Decode
        for t in range(1, trg_len):
            output, h, c, attention = self.decoder(decoder_input, h, c, encoder_outputs)
            predictions.append(output)
            all_attention.append(attention)
            
            # Teacher forcing
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                decoder_input = trg[:, t, :]
            else:
                decoder_input = output
        
        predictions = torch.stack(predictions, dim=1)  # [batch, trg_len-1, output_dim]
        all_attention = torch.stack(all_attention, dim=1)  # [batch, trg_len-1, src_len]
        
        return predictions, all_attention


In [ ]:
# =====================================================================
# DEMONSTRATION AND TESTING
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Hyperparameters
    BATCH_SIZE = 8
    SRC_LEN = 20
    TRG_LEN = 15
    INPUT_DIM = 64
    OUTPUT_DIM = 32
    HIDDEN_DIM = 128
    
    # Create models
    print("=" * 60)
    print("BAHDANAU ATTENTION SEQ2SEQ")
    print("=" * 60)
    model_bahdanau = Seq2Seq(INPUT_DIM, OUTPUT_DIM, HIDDEN_DIM, 
                             attention_type='bahdanau', device=device).to(device)
    
    print("\n" + "=" * 60)
    print("LUONG ATTENTION SEQ2SEQ")
    print("=" * 60)
    model_luong = Seq2Seq(INPUT_DIM, OUTPUT_DIM, HIDDEN_DIM, 
                          attention_type='luong', device=device).to(device)
    
    # Generate mock data
    src = torch.randn(BATCH_SIZE, SRC_LEN, INPUT_DIM).to(device)
    trg = torch.randn(BATCH_SIZE, TRG_LEN, INPUT_DIM).to(device)
    
    print(f"\nSource shape: {src.shape}")
    print(f"Target shape: {trg.shape}\n")
    
    # Forward pass
    model_bahdanau.eval()
    model_luong.eval()
    
    with torch.no_grad():
        pred_bahdanau, attn_bahdanau = model_bahdanau(src, trg, teacher_forcing_ratio=0.0)
        pred_luong, attn_luong = model_luong(src, trg, teacher_forcing_ratio=0.0)
    
    print(f"✓ Bahdanau Predictions shape: {pred_bahdanau.shape}")
    print(f"✓ Bahdanau Attention shape: {attn_bahdanau.shape}")
    print(f"\n✓ Luong Predictions shape: {pred_luong.shape}")
    print(f"✓ Luong Attention shape: {attn_luong.shape}")
    
    print(f"\n\n✅ Seq2Seq with Attention models working correctly!")
